В этом ноутбуке готовим данные для MLP через OHE

In [1]:
import pandas as pd
import numpy as np

"""
X_train = pd.read_csv("X_train.csv")
X_val = pd.read_csv("X_val.csv")
X_test = pd.read_csv("X_test.csv")

y_train = pd.read_csv("y_train.csv")["target_log_price"]
y_val = pd.read_csv("y_val.csv")["target_log_price"]
y_test = pd.read_csv("y_test.csv")["target_log_price"]

train_df = pd.read_csv("train_df.csv")
val_df = pd.read_csv("val_df.csv")
test_df = pd.read_csv("test_df.csv")

final_dataset = pd.read_csv("final_dataset_before_split.csv")
"""

X_train = pd.read_csv("outputs/X_train.csv")
X_val = pd.read_csv("outputs/X_val.csv")
X_test = pd.read_csv("outputs/X_test.csv")

y_train = pd.read_csv("outputs/y_train.csv")["target_log_price"]
y_val = pd.read_csv("outputs/y_val.csv")["target_log_price"]
y_test = pd.read_csv("outputs/y_test.csv")["target_log_price"]

train_df = pd.read_csv("outputs/train_df.csv")
val_df = pd.read_csv("outputs/val_df.csv")
test_df = pd.read_csv("outputs/test_df.csv")

final_dataset = pd.read_csv("outputs/final_dataset_before_split.csv")


print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (3698, 45)
X_val: (793, 45)
X_test: (793, 45)
y_train: (3698,)
y_val: (793,)
y_test: (793,)


In [2]:
all_col = final_dataset.columns
all_col

Index(['brand', 'model', 'year', 'body_type', 'fuel_type', 'engine_volume',
       'mileage', 'transmission', 'drive_type', 'steering_wheel',
       'kz_registration', 'imgs_count', 'price', 'g_start_year', 'g_end_year',
       'g_is_current', 'g_number', 'g_is_restyling', 'g_age_car_release',
       'g_span', 'generation_number_missing', 'target_log_price', 'is_premium',
       'city_name', 'region', 'city_tier', 'is_big_city', 'is_electric',
       'car_age', 'is_new_car', 'is_recent_car', 'is_old_car', 'age_bucket',
       'is_metallic', 'base_color', 'color_known', 'log_mileage',
       'mileage_per_year', 'log_mileage_per_year', 'has_low_mileage',
       'has_high_mileage', 'is_automatic_like', 'body_segment',
       'log_imgs_count', 'has_few_photos', 'has_many_photos', 'brand_model'],
      dtype='object')

Выделяем категориальные признаки

In [3]:
cat_col = ['brand', 'model', 'body_type', 'fuel_type', 'transmission', 'drive_type', 'city_name', 'region', 'city_tier', 'age_bucket', 'base_color', 'body_segment',]

brand_model - уберем признак, потому что он статистически вдет себя как model

In [4]:
X_train = X_train.drop(columns=["brand_model"])
X_val = X_val.drop(columns=["brand_model"])
X_test = X_test.drop(columns=["brand_model"])

Посмотрим кол-во уникальных значений в train для каждого категориального признака. Признаки с небольшим кол-вом значений можно кодировать через OHE без дополнительной группировки. Признаки с большим количеством категорий или большим числом редких значений нужно предварительно сгруппировать, чтобы не создавать излишнего кол-ва OHE-колонок

In [5]:
for i in cat_col:
    print(i, "уникальных в train", X_train[i].nunique())

brand уникальных в train 8
model уникальных в train 346
body_type уникальных в train 14
fuel_type уникальных в train 6
transmission уникальных в train 4
drive_type уникальных в train 3
city_name уникальных в train 94
region уникальных в train 17
city_tier уникальных в train 4
age_bucket уникальных в train 6
base_color уникальных в train 21
body_segment уникальных в train 6


brand, fuel_type, transmission, drive_type, city_tier, age_bucket, body_segment - имеют небольшое кол-во уникальных значений, их можно оставить без  группировки.

model, city_name, base_color и body_type - имеют большое кол-во уникальных значений, для них првоерим частоты и занесем редкие в отдельные группы

In [6]:
for i in cat_col:
    in_train = set(X_train[i].unique())
    in_val = set(X_val[i].unique())
    in_test = set(X_test[i].unique())

    diff_val = in_val - in_train
    diff_test = in_test - in_train

    print(i, "val", len(diff_val))
    print(i, "test", len(diff_test))

brand val 0
brand test 0
model val 21
model test 23
body_type val 0
body_type test 0
fuel_type val 0
fuel_type test 0
transmission val 0
transmission test 0
drive_type val 0
drive_type test 0
city_name val 9
city_name test 7
region val 0
region test 0
city_tier val 0
city_tier test 0
age_bucket val 0
age_bucket test 0
base_color val 0
base_color test 0
body_segment val 0
body_segment test 0


Тут мы видим, что у model и city_name в val и test есть значенияб которых нет в train. Поэтому их нужно буедт закодировать.

делем группировку редких значений по train. Тут мы частоты признаокв считаем только на X_train, признаки, которые встречаются в train меньше 10 раз, заменяем на Other, к val и test применяем то, что сделали к train? признаки из val/test, которых не было в train идут в Other

Так мы уменьшаем размерность OHE

In [7]:
models = X_train["model"].value_counts()
m_ok = models[models >= 10].index

X_train["m_other"] = "Other"
X_val["m_other"] = "Other"
X_test["m_other"] = "Other"

X_train.loc[X_train["model"].isin(m_ok), "m_other"] = X_train["model"]
X_val.loc[X_val["model"].isin(m_ok), "m_other"] = X_val["model"]
X_test.loc[X_test["model"].isin(m_ok), "m_other"] = X_test["model"]

In [8]:
colors = X_train["base_color"].value_counts()
c_ok = colors[colors >= 10].index

X_train["c_other"] = "Other"
X_val["c_other"] = "Other"
X_test["c_other"] = "Other"

X_train.loc[X_train["base_color"].isin(c_ok), "c_other"] = X_train["base_color"]
X_val.loc[X_val["base_color"].isin(c_ok), "c_other"] = X_val["base_color"]
X_test.loc[X_test["base_color"].isin(c_ok), "c_other"] = X_test["base_color"]

In [9]:
bodies = X_train["body_type"].value_counts()
b_ok = bodies[bodies >= 10].index

X_train["b_other"] = "Other"
X_val["b_other"] = "Other"
X_test["b_other"] = "Other"

X_train.loc[X_train["body_type"].isin(b_ok), "b_other"] = X_train["body_type"]
X_val.loc[X_val["body_type"].isin(b_ok), "b_other"] = X_val["body_type"]
X_test.loc[X_test["body_type"].isin(b_ok), "b_other"] = X_test["body_type"]

In [10]:
cities = X_train["city_name"].value_counts()
cc_ok = cities[cities >= 10].index

X_train["cc_other"] = "Other"
X_val["cc_other"] = "Other"
X_test["cc_other"] = "Other"

X_train.loc[X_train["city_name"].isin(cc_ok), "cc_other"] = X_train["city_name"]
X_val.loc[X_val["city_name"].isin(cc_ok), "cc_other"] = X_val["city_name"]
X_test.loc[X_test["city_name"].isin(cc_ok), "cc_other"] = X_test["city_name"]

In [11]:
print(X_train["model"].nunique(), X_train["m_other"].nunique())
print(X_train["city_name"].nunique(), X_train["cc_other"].nunique())
print(X_train["base_color"].nunique(), X_train["c_other"].nunique())
print(X_train["body_type"].nunique(), X_train["b_other"].nunique())

346 84
94 21
21 16
14 12


In [12]:
other_col = ["m_other", "cc_other", "c_other", "b_other"]

for i in other_col:
    print(i, "train", X_train[i].nunique(), "val", X_val[i].nunique(), "test", X_test[i].nunique())

m_other train 84 val 83 test 82
cc_other train 21 val 21 test 20
c_other train 16 val 16 test 16
b_other train 12 val 12 test 12


Финальные группы признаков

Перед кодированием разделяем признаки на три группы:

num_col - числовые признаки, которые нужно масштабировать

bin_col - бинарные признаки, которые уже в формате 0/1

col_ohe - категориальные признаки, которые нужно закодировать через OHE


In [13]:
all_col

Index(['brand', 'model', 'year', 'body_type', 'fuel_type', 'engine_volume',
       'mileage', 'transmission', 'drive_type', 'steering_wheel',
       'kz_registration', 'imgs_count', 'price', 'g_start_year', 'g_end_year',
       'g_is_current', 'g_number', 'g_is_restyling', 'g_age_car_release',
       'g_span', 'generation_number_missing', 'target_log_price', 'is_premium',
       'city_name', 'region', 'city_tier', 'is_big_city', 'is_electric',
       'car_age', 'is_new_car', 'is_recent_car', 'is_old_car', 'age_bucket',
       'is_metallic', 'base_color', 'color_known', 'log_mileage',
       'mileage_per_year', 'log_mileage_per_year', 'has_low_mileage',
       'has_high_mileage', 'is_automatic_like', 'body_segment',
       'log_imgs_count', 'has_few_photos', 'has_many_photos', 'brand_model'],
      dtype='object')

In [14]:
col_ohe = [
    "brand",
    "m_other",
    "b_other",
    "body_segment",
    "fuel_type",
    "transmission",
    "drive_type",
    "cc_other",
    "region",
    "city_tier",
    "age_bucket",
    "c_other"
]

num_col = [
    "year",
    "engine_volume",
    "mileage",
    "imgs_count",
    "g_start_year",
    "g_end_year",
    "g_number",
    "g_age_car_release",
    "g_span",
    "car_age",
    "log_mileage",
    "mileage_per_year",
    "log_mileage_per_year",
    "log_imgs_count"
]

bin_col = [
    "steering_wheel",
    "kz_registration",
    "g_is_current",
    "g_is_restyling",
    "generation_number_missing",
    "is_premium",
    "is_big_city",
    "is_electric",
    "is_new_car",
    "is_recent_car",
    "is_old_car",
    "is_metallic",
    "color_known",
    "has_low_mileage",
    "has_high_mileage",
    "is_automatic_like",
    "has_few_photos",
    "has_many_photos"
]

In [15]:
all_cols = col_ohe + num_col + bin_col

print(len(all_cols))

44


Всего признаков до OHE: 44

In [16]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [17]:
scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train[num_col])
X_val_num = scaler.transform(X_val[num_col])
X_test_num = scaler.transform(X_test[num_col])

In [18]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = ohe.fit_transform(X_train[col_ohe])
X_val_cat = ohe.transform(X_val[col_ohe])
X_test_cat = ohe.transform(X_test[col_ohe])

In [19]:
X_train_bin = X_train[bin_col].values
X_val_bin = X_val[bin_col].values
X_test_bin = X_test[bin_col].values

In [20]:
X_train_ohe = np.hstack([X_train_num, X_train_bin, X_train_cat])
X_val_ohe = np.hstack([X_val_num, X_val_bin, X_val_cat])
X_test_ohe = np.hstack([X_test_num, X_test_bin, X_test_cat])

print(X_train_ohe.shape)
print(X_val_ohe.shape)
print(X_test_ohe.shape)

(3698, 219)
(793, 219)
(793, 219)


In [21]:
ohe_fe = ohe.get_feature_names_out(col_ohe)

fe = (num_col+ bin_col+ list(ohe_fe))

print(len(fe))
print(X_train_ohe.shape)

219
(3698, 219)


После OHE количество признаков увеличилось до 219.

Сохраняем список fe, чтобы понимать, какая колонка финальной матрицы за что отвечает


Кол-во признаков: 219
Кол-во колонок в X_train_ohe: 219

In [22]:
import os

OUTPUT_DIR = "mlp_data_ohe"
os.makedirs(OUTPUT_DIR, exist_ok=True)

pd.DataFrame(X_train_ohe, columns=fe).to_csv(f"{OUTPUT_DIR}/X_train_ohe.csv", index=False)
pd.DataFrame(X_val_ohe, columns=fe).to_csv(f"{OUTPUT_DIR}/X_val_ohe.csv", index=False)
pd.DataFrame(X_test_ohe, columns=fe).to_csv(f"{OUTPUT_DIR}/X_test_ohe.csv", index=False)
y_train.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_val.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_val.csv", index=False)
y_test.to_frame("target_log_price").to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)
pd.Series(fe).to_csv(f"{OUTPUT_DIR}/feature_names_ohe.csv", index=False)